# Baseline — generacja apelacji

Najnaiwniejsze podejście: **wszystkie akta w jeden prompt** + „napisz apelację”.
Ten notebook tylko **generuje** apelację (i liczy koszt). Ewaluację pokazuje
osobny notebook `eval_walkthrough.ipynb`.

> Notebook używa taniego `gpt-5.4-mini` (demo). W `__main__` modułu (`python -m baseline.main`) generacja i ocena idą modelem z `.env` (`gpt-5.4`).

## 0. Konfiguracja

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini"  # w notebookach tani model — to tylko demo
print("model:", os.environ["LLM_MODEL"], "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

model: gpt-5.4-mini | klucz z .env: True


## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt — policzmy, ile to tokenów.

In [2]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)
print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie: ~{prompt_tokens:,}")

Dokumentów: 16
Znaków w kontekście: 52,159
Tokenów w promptcie: ~19,341


## 2. Generowanie apelacji (jeden prompt) + koszt

In [3]:
from baseline.main import generate_appeal
from src.llm import track_usage
from src.cost import cost_summary

with track_usage() as u:
    appeal = generate_appeal(docs).tekst

print(f"Apelacja: {len(appeal):,} znaków")
print(f"Koszt generacji: {cost_summary(u, os.environ['LLM_MODEL'])}\n")
print(appeal[:1000], "...")

Apelacja: 6,567 znaków
Koszt generacji: gpt-5.4-mini: 1 wywołań, 19,473 wej + 2,304 wyj tok = $0.0250 (≈ $0.02497/wyw.) | 15.5s (≈15.5s/wyw.)

Sąd Okręgowy w Krakowie
za pośrednictwem
Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
II Wydział Karny

Sygn. akt II K 25/25

Obrońca oskarżonego Daniela Dzika
radca prawny Lidia Lis

APELACJA
od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
z dnia 14 marca 2025 r., sygn. akt II K 25/25

Działając w imieniu oskarżonego Daniela Dzika, na podstawie art. 425 § 1 i 2 k.p.k., art. 444 k.p.k. oraz art. 427 § 1 i 2 k.p.k., zaskarżam powyższy wyrok w całości, tj. w zakresie punktów 1–9.

Zaskarżonemu wyrokowi zarzucam:

I. na podstawie art. 438 pkt 2 k.p.k. – obrazę przepisów postępowania, która mogła mieć wpływ na treść orzeczenia, a mianowicie:
1) art. 7 k.p.k. w zw. z art. 410 k.p.k. i art. 424 § 1 pkt 1 k.p.k., poprzez dowolną, a nie swobodną ocenę materiału dowodowego, polegającą na przyjęciu, że oskarżony prowadził pojazd mechaniczny

## 3. Zapis apelacji

Do `data/output/baseline/` (apelacja + log danego przebiegu razem; poza gitem).

In [4]:
from src.output import save_appeal

path = save_appeal(appeal, "baseline")
print("Zapisano:", path)

Zapisano: /Users/ewasuknarowska/Projects/WomenInTech/baseline/output/apelacja_2026-06-07_094819.txt


## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem. Szybkie, ale: zapchany kontekst, „lost in the middle”, brak śladu rozumowania, podatność na pominięcia.

Jak dobra jest ta apelacja? → `eval_walkthrough.ipynb` (pokrycie + jakość).